# Operation Ditwah – Crisis Intelligence Pipeline 
 
**Course Module**: Week 01 – Prompt Engineering Essentials

**Scenario**: Post-Cyclone Ditwah Relief (Sri Lanka)

**Name**: R.A.H.P. Ranasinghe

**Student No.**: 546

![operation ditwah banner](../data/img/cover.png)

In [18]:
# Import libraries
import sys
sys.path.append('..')

from utils.token_utils import pick_encoding, count_text_tokens
from utils.logging_utils import log_llm_call
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.router import pick_model
import tiktoken
import pandas as pd

## Part 1: The ”Contract” & Few-Shot Learning (20 Points) 
**The Problem:** 
The classifier mistakes ”News Reports” for ”SOS Calls”. 


**Your Task:**

- Load  data/sample_messages.txt.  This  file  contains  a  batch  of  mixed  incoming 
messages (SOS, News, Status Updates). 
- Refactor the classification prompt using skeleton.v1. 
- Constraint: You  MUST use few-shot prompting.  Provide  at  least  4 labelled 
examples (Rescue, Supply, Info, Other) inside the prompt to teach the model. 
- OutputContract: District: [Name] | Intent: [Category] | Priority: [High/Low] 
- Process Loop: Iterate  through  the  loaded  messages,  classify  each  one  using 
your few-shot prompt, and store the results. 

**Deliverable:**
Save the above classified results into a CSV/Excel file using pandas named 
output/classified_messages.xlsx.

**Success Check:**
- Input Example: ”Breaking News: Kelani River level at 9m.” 
  - Expected Output: District: Colombo | Intent: Info | Priority: Low 

- Input Example: ”We are trapped on the roof with 3 kids!” 
  - Expected Output: District: None | Intent: Rescue | Priority: High

In [26]:
model = pick_model('groq', 'general')
client = LLMClient('groq', model=model)

In [27]:
# Load data
with open('../data/Sample Messages.txt', 'r') as f:
    sample_messages = list(filter(None,f.read().splitlines()))

sample_messages[:5]


['BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued.',
 'SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.',
 'Update: Kandy road cleared near Peradeniya. Traffic moving slowly. No victims reported.',
 'Does anyone have extra dry rations for the camp in Gampaha?',
 'News just in: Kelani river water level is at 7ft.']

In [28]:
len(sample_messages)

50

In [46]:
def classify_message(msg):
    examples = '''
        Input: "BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued."
        Output: "District: Colombo | Intent: Info | Priority: Low"

        Input: "Urgent! Flooding in Galle. People trapped in houses. Need immediate rescue!"
        Output: "District: Galle | Intent: Rescue | Priority: High"

        Input: "Food and water supplies running low in Jaffna. Requesting urgent supply drop."
        Output: "District: Jaffna | Intent: Supply | Priority: High"

        Input: "Where are you right now?"
        Output: "District: None | Intent: Other | Priority: Low"

        Input: "8 people trapped in a collapsed building. Rescue teams required immediately."
        Output: "District: None | Intent: Rescue | Priority: High"
    '''

    prompt_text, spec = render(
        'few_shot.v1',
        role='Crisis message classifier',
        examples=examples,
        query= f'Classify the following message: "{msg}"',
        constraints='Respond only with the extracted information. Do not include any explanations or additional text.',
        format='District: [Name or None] | Intent: [Category] | Priority: [High/Low]'
    )

    messages = [{ "role": "user", "content": prompt_text }]
    response = client.chat(messages, temperature=0.0, max_tokens=100)
    
    log_llm_call('groq', model, 'few_shot', response['latency_ms'], response['usage'])

    output = response['text']
    print(f'Processed: "{msg}" --> {output}')
    
    return{
        "message": msg,
        "district": output.split('|')[0].split(':')[1].strip(),
        "intent": output.split('|')[1].split(':')[1].strip(),
        "priority": output.split('|')[2].split(':')[1].strip(),
        "output": response['text'],
        "raw_response": response
    }

In [48]:
results = []

for msg in sample_messages:
    res = classify_message(msg)
    results.append(res)

Processed: "BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued." --> District: Colombo | Intent: Info | Priority: Low
Processed: "SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately." --> District: Gampaha | Intent: Rescue | Priority: High
Processed: "Update: Kandy road cleared near Peradeniya. Traffic moving slowly. No victims reported." --> District: Kandy | Intent: Info | Priority: Low
Processed: "Does anyone have extra dry rations for the camp in Gampaha?" --> District: Gampaha | Intent: Supply | Priority: Low
Processed: "News just in: Kelani river water level is at 7ft." --> District: Colombo | Intent: Info | Priority: Low
Processed: "My uncle is stuck in the tree (immediate danger)." --> District: None | Intent: Rescue | Priority: High
Processed: "Just saw on news that Gampaha town is flooded. Hope everyone is safe." --> District: Gampaha | Intent: Info | Priority: Low
Processed: "We ar

In [49]:
print(results[:5])
print(len(results))

[{'message': 'BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued.', 'district': 'Colombo', 'intent': 'Info', 'priority': 'Low', 'output': 'District: Colombo | Intent: Info | Priority: Low', 'raw_response': {'text': 'District: Colombo | Intent: Info | Priority: Low', 'usage': {'input_tokens_est': 283, 'context_tokens_est': 0, 'total_est': 286, 'prompt_tokens_actual': 316, 'completion_tokens_actual': 13, 'total_tokens_actual': 329}, 'latency_ms': 161, 'raw': ChatCompletion(id='chatcmpl-387dbba4-cce9-4143-a144-f620000b6da2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='District: Colombo | Intent: Info | Priority: Low', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1774188281, model='llama-3.1-8b-instant', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_4387d3ed

In [50]:
df = pd.DataFrame(results)
print(df.head())
df.to_excel('../output/classified_messages.xlsx', index=False)
print("\nResults saved to ../output/classified_messages.xlsx")

                                             message district  intent  \
0  BREAKING: Water levels in Kelani River (Colomb...  Colombo    Info   
1  SOS: 5 people trapped on a roof in Ja-Ela (Gam...  Gampaha  Rescue   
2  Update: Kandy road cleared near Peradeniya. Tr...    Kandy    Info   
3  Does anyone have extra dry rations for the cam...  Gampaha  Supply   
4  News just in: Kelani river water level is at 7ft.  Colombo    Info   

  priority                                             output  \
0      Low   District: Colombo | Intent: Info | Priority: Low   
1     High  District: Gampaha | Intent: Rescue | Priority:...   
2      Low     District: Kandy | Intent: Info | Priority: Low   
3      Low  District: Gampaha | Intent: Supply | Priority:...   
4      Low   District: Colombo | Intent: Info | Priority: Low   

                                        raw_response  
0  {'text': 'District: Colombo | Intent: Info | P...  
1  {'text': 'District: Gampaha | Intent: Rescue |...  
2  {'

In [52]:
# Success check
test_cases = ["Breaking News: Kelani River level at 9m.", "We are trapped on the roof with 3 kids!"]
test_results = []

for msg in test_cases:
    result = classify_message(msg)

Processed: "Breaking News: Kelani River level at 9m." --> District: Colombo | Intent: Info | Priority: Low
Processed: "We are trapped on the roof with 3 kids!" --> District: None | Intent: Rescue | Priority: High
